# Дополнительное задание: спутниковая океанология

**Цель работы:** исследовать метеорологические наблюдения за сентябрь и построить карту температуры поверхности океана по спутниковому набору данных.

В работе используются два файла:
- `September.xlsx` — временные ряды температуры воздуха и атмосферного давления;
- `sst.nc` — глобальный набор данных температуры поверхности моря NOAA OISST.

> **Важное замечание о единицах измерения:** в приложенном файле `sst.nc` переменная `sst` уже имеет единицы `Celsius` (градусы Цельсия), а не Кельвины. Поэтому повторно вычитать 273.15 нельзя. В коде ниже единицы проверяются автоматически: перевод из Кельвинов выполняется только в том случае, если файл действительно содержит температуру в K.

## 0. Подготовка среды

Если необходимые библиотеки ещё не установлены, выполните в отдельной ячейке Jupyter:

```python
%pip install pandas matplotlib openpyxl xarray netCDF4
```

Поместите файлы `September.xlsx` и `sst.nc` в ту же папку, что и этот ноутбук.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["font.size"] = 11

## Задание 1. Временной ход температуры воздуха и атмосферного давления

Загрузим таблицу, посмотрим на её структуру и отсортируем наблюдения по времени. Отдельные графики удобнее для чтения, поскольку температура и давление имеют разные единицы измерения и масштабы.

In [ ]:
# Чтение исходной таблицы
weather = pd.read_excel("September.xlsx")

# Первые строки и общая информация о таблице
display(weather.head())
weather.info()

In [ ]:
# Преобразуем время в формат datetime и упорядочим наблюдения
weather["Time"] = pd.to_datetime(weather["Time"])
weather = weather.sort_values("Time")

# Проверим диапазон времени и наличие пропусков
print("Период наблюдений:", weather["Time"].min(), "—", weather["Time"].max())
print("\\nКоличество пропусков:")
display(weather[["Time", "Temperature", "Pressure"]].isna().sum())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(weather["Time"], weather["Temperature"], marker=".", linewidth=1.5)
axes[0].set_title("Изменение температуры воздуха во времени")
axes[0].set_ylabel("Температура, °C")
axes[0].grid(True, alpha=0.3)

axes[1].plot(weather["Time"], weather["Pressure"], marker=".", linewidth=1.5)
axes[1].set_title("Изменение атмосферного давления во времени")
axes[1].set_ylabel("Давление")
axes[1].set_xlabel("Время")
axes[1].grid(True, alpha=0.3)

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

### Краткое описание результата

На первом графике показано изменение температуры воздуха в течение периода наблюдений, на втором — изменение атмосферного давления. Графики построены отдельно, чтобы различия в численных масштабах величин не мешали сравнению их временного хода.

Единица измерения давления в исходной таблице явно не указана, поэтому в подписи оси она не предполагается.

## Задание 2. Пространственное распределение температуры поверхности моря

Откроем NetCDF-файл с помощью `xarray`. Сначала выведем структуру набора данных, список переменных, размерности и единицы измерения температуры.

In [ ]:
ds = xr.open_dataset("sst.nc")
display(ds)

In [ ]:
print("Переменные и их характеристики:")
for name, variable in ds.data_vars.items():
    print(f"\\n{name}")
    print("  Размерности:", variable.dims)
    print("  Размер массива:", variable.shape)
    print("  Единицы измерения:", variable.attrs.get("units", "не указаны"))
    print("  Описание:", variable.attrs.get("long_name", "не указано"))

print("\\nКоординаты:")
for name, coordinate in ds.coords.items():
    print(f"{name}: размер {coordinate.shape}, единицы {coordinate.attrs.get('units', 'не указаны')}")

### Проверка единиц измерения и перевод температуры

В этом наборе данных температура поверхности моря хранится в переменной `sst`. В приложенном файле её атрибут `units` равен `Celsius`, а значения закодированы с коэффициентом масштаба `0.01`. `xarray` при стандартном открытии файла автоматически применяет `scale_factor` и обрабатывает значения пропусков.

Поэтому для данного файла дополнительный перевод не требуется. На случай, если будет использован другой файл с температурой в Кельвинах, ниже предусмотрена проверка единиц.

In [ ]:
sst = ds["sst"]
units = str(sst.attrs.get("units", "")).strip().lower()

if units in ["k", "kelvin", "kelvins"]:
    sst_c = sst - 273.15
    sst_c.attrs["units"] = "°C"
    print("Температура была переведена из Кельвинов в градусы Цельсия.")
elif units in ["celsius", "degree_celsius", "degrees_celsius", "°c", "degc"]:
    sst_c = sst
    print("В файле температура уже задана в градусах Цельсия. Перевод не требуется.")
else:
    raise ValueError(f"Не удалось определить единицы измерения температуры: {units!r}")

print("Размерности температуры:", sst_c.dims)
print("Размер массива:", sst_c.shape)
print("Единицы измерения:", sst_c.attrs.get("units", units))

### Выбор области океана

Для построения подробной карты выберем прямоугольную область северной части Атлантического океана:
- широта: от 10° до 60° северной широты;
- долгота: от 280° до 340° восточной долготы.

В файле долгота задана в диапазоне от 0° до 360°, поэтому выбранный интервал соответствует примерно 80°–20° западной долготы.

In [ ]:
# Убираем измерение глубины zlev (в наборе оно содержит один уровень)
sst_surface = sst_c.isel(zlev=0)

# Выбираем первый (и единственный) временной срез
sst_day = sst_surface.isel(time=0)

# Прямоугольная область Северной Атлантики
lat_min, lat_max = 10, 60
lon_min, lon_max = 280, 340

region = sst_day.sel(
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max)
)

print("Размер выбранного участка:", region.shape)
print("Широта:", float(region.lat.min()), "—", float(region.lat.max()))
print("Долгота:", float(region.lon.min()), "—", float(region.lon.max()))

In [ ]:
# Определяем дату из временной координаты
date_value = pd.to_datetime(ds["time"].values[0]).strftime("%Y-%m-%d")

# Карта температуры выбранной области
plt.figure(figsize=(12, 6))
mesh = plt.pcolormesh(
    region["lon"],
    region["lat"],
    region,
    shading="auto",
    cmap="coolwarm"
)

plt.colorbar(mesh, label="Температура поверхности моря, °C")
plt.xlabel("Долгота, ° восточной долготы")
plt.ylabel("Широта, ° северной широты")
plt.title(f"Температура поверхности Северной Атлантики — {date_value}")
plt.grid(True, linestyle="--", alpha=0.35)
plt.tight_layout()
plt.show()

### Вывод по заданию 2

Файл `sst.nc` содержит глобальное поле температуры поверхности моря с пространственным шагом 0.25°. Переменная `sst` имеет размерности времени, уровня, широты и долготы. В предоставленном файле представлен один временной срез — **1 сентября 1981 года**.

Для выбранной области Северной Атлантики построена карта распределения температуры. Цветовая шкала показывает температуру в градусах Цельсия. Поскольку исходные единицы переменной `sst` — `Celsius`, преобразование из Кельвинов в этом конкретном наборе данных не выполнялось.

In [ ]:
# Закрываем файл после завершения работы
ds.close()